# 🐄 Cattle Analytics — Colab Pro+ Experiment Notebook
**Pipeline**: YOLOv8 Detection → ByteTrack Tracking → MobileNetV3 Behavior Classification

**Data layout on Drive**:
```
MyDrive/cattle-analytics/data/
├── v1/   ← first 10 videos go here
│   ├── <video>.mp4
│   └── annotations.xml
├── v2/ ... v10/
```

## Runtime
- **Always use**: Runtime → Change runtime type → **A100 GPU** (Colab Pro+)
- A100 trains YOLOv8s ~4× faster than T4

## Workflow
Run cells **top to bottom**, in order. Each section is self-contained and skippable if already done.

## 📦 Cell 1 — Install Dependencies
> Run once per session. Subsequent runs are fast (cached by pip).

In [ ]:
%%capture
!pip install ultralytics==8.3.0
!pip install supervision==0.21.0    # ByteTrack wrapper
!pip install lap                     # ByteTrack dependency
!pip install albumentations
!pip install timm
!pip install seaborn scikit-learn
!pip install tabulate

import torch
print(f'CUDA available : {torch.cuda.is_available()}')
print(f'GPU            : {torch.cuda.get_device_name(0)}')
print(f'VRAM           : {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')

## ☁️ Cell 2 — Mount Drive & Pull Code
> Clones your experiment repo from GitHub (first time) or pulls updates.

In [ ]:
from google.colab import drive
drive.mount('/content/drive', force_remount=False)

import os, sys

# ── CONFIGURE THESE ──────────────────────────────────────────────────────────
DRIVE_ROOT  = '/content/drive/MyDrive/cattle-analytics'
GITHUB_REPO = 'https://github.com/YOUR_ORG/cattle-analytics.git'   # ← update
CODE_DIR    = f'{DRIVE_ROOT}/experiment'
# ─────────────────────────────────────────────────────────────────────────────

os.makedirs(DRIVE_ROOT, exist_ok=True)

if os.path.exists(f'{DRIVE_ROOT}/.git'):
    print('Repo already cloned. Pulling latest...')
    !cd {DRIVE_ROOT} && git pull
else:
    print('Cloning repo...')
    !git clone {GITHUB_REPO} {DRIVE_ROOT}

# Add experiment/ to Python path
sys.path.insert(0, CODE_DIR)
os.chdir(CODE_DIR)
print(f'Working dir: {os.getcwd()}')

## 🗂️ Cell 3 — Setup Drive Paths & Discover Version Folders
> Verifies your data is in the right place.  
> Expected: `data/v1/`, `data/v2/`, … each with `.mp4` + `annotations.xml`

In [ ]:
# Runs 00_setup_drive.py — sets up all path constants and discovers version folders
exec(open('scripts/00_setup_drive.py').read())

print(f'\n📁 Discovered {len(VERSION_FOLDERS)} version folders: {[v.name for v in VERSION_FOLDERS]}')

## 🎞️ Cell 4 — Extract Annotated Frames
> Reads XML to find which frame IDs are annotated → extracts + resizes to 1280×720.  
> Output: `processed/frames/v1/<stem>/frame_XXXXXX.jpg`
>
> ⏱️ Estimated: ~2–5 min for 10 videos on A100 (I/O bound)

In [ ]:
!python scripts/01_extract_frames.py \
    --max_versions 10 \
    --width 1280 \
    --height 720

## 🏷️ Cell 5 — Convert CVAT XML → YOLO Format
> Creates YOLO labels + `dataset.yaml`. Train/val split is done at **video level**  
> to prevent temporal leakage (frames from the same video stay in the same split).

In [ ]:
!python scripts/02_cvat_to_yolo.py \
    --val_split 0.2 \
    --seed 42

## ✂️ Cell 6 — Generate Behavior Crops (Classifier Training Data)
> Crops each annotated calf per behavior class → `processed/behavior_crops/<behavior>/`  
> Adds 15% padding around bounding boxes for context.

In [ ]:
!python scripts/03_generate_crops.py --padding 0.15

## 🔍 Cell 7 — Dataset Health Check
> **Always run this before training.**  
> Shows frame counts, behavior distribution, class imbalance warnings, and label integrity.

In [ ]:
!python scripts/04_check_dataset_stats.py

## 🖼️ Cell 8 — Visualize Annotations (Optional Sanity Check)
> Draws bounding boxes + behavior labels onto sampled frames.  
> Great for catching annotation errors before training.

In [ ]:
!python scripts/05_visualize_annotations.py --version v1 --max_frames 10

# Show a sample image inline
from IPython.display import Image as IPyImage
import glob
samples = glob.glob('processed/viz/v1/*.jpg')[:3]
for s in samples:
    display(IPyImage(s, width=800))

## 🏋️ Cell 9 — Train YOLOv8 Calf Detector
> Trains YOLOv8s on your calf detection dataset.  
> Logs metrics to `logs/experiments/detector_<timestamp>/metrics.json`  
>
> ⏱️ Estimated: ~30–60 min on A100 for 100 epochs with 10 videos
>
> **💡 Tips**:
> - Start with `--epochs 30` for a quick smoke test first
> - Use `--model yolov8n.pt` for faster iteration; upgrade to `yolov8s.pt` for final run

In [ ]:
# Quick smoke test (30 epochs)
!python train/train_detector.py \
    --model yolov8s.pt \
    --epochs 30 \
    --batch 8 \
    --imgsz 1280

In [ ]:
# Full training run (100 epochs)
# Uncomment when satisfied with smoke test results
# !python train/train_detector.py \
#     --model yolov8s.pt \
#     --epochs 100 \
#     --batch 8 \
#     --imgsz 1280

## 🧠 Cell 10 — Train Behavior Classifier
> Trains MobileNetV3-Small on the behavior crops.  
> Uses class-weighted loss + WeightedRandomSampler to handle imbalance.
>
> ⚠️ **Prerequisite**: Need ≥200 crops per class. Check Cell 7 output first.
>
> ⏱️ Estimated: ~10–20 min on A100 for 50 epochs

In [ ]:
!python train/train_classifier.py \
    --epochs 50 \
    --batch 64 \
    --lr 1e-4

## 📊 Cell 11 — Evaluate & Compare All Experiments
> Generates confusion matrix + per-class report, then prints experiment comparison table.

In [ ]:
# Summary table of all experiments
!python evaluation/eval_pipeline.py --phase summary

In [ ]:
# Full classifier evaluation (most recent checkpoint auto-detected)
!python evaluation/eval_pipeline.py --phase classifier

# Show confusion matrix
from IPython.display import Image as IPyImage
import glob
cms = sorted(glob.glob('models/checkpoints/classifier_*/confusion_matrix.png'))
if cms:
    display(IPyImage(cms[-1], width=700))

## 🎬 Cell 12 — Run Full Inference Pipeline on a Video
> Detect → Track → Classify → Annotated video + JSON output

In [ ]:
import glob, os

# Auto-select latest detector and classifier
detectors   = sorted(glob.glob('models/checkpoints/detector_*/weights/best.pt'))
classifiers = sorted(glob.glob('models/checkpoints/classifier_*/best_classifier.pth'))

DETECTOR_PATH   = detectors[-1]   if detectors   else None
CLASSIFIER_PATH = classifiers[-1] if classifiers else None

# ← Set this to any video from your Drive
INPUT_VIDEO  = 'data/v1/<your_video>.mp4'    # UPDATE THIS
OUTPUT_VIDEO = 'data/v1/output_annotated.mp4'

print(f'Detector   : {DETECTOR_PATH}')
print(f'Classifier : {CLASSIFIER_PATH}')
print(f'Input      : {INPUT_VIDEO}')

if DETECTOR_PATH and CLASSIFIER_PATH and os.path.exists(INPUT_VIDEO):
    !python inference/pipeline.py \
        --video      {INPUT_VIDEO} \
        --detector   {DETECTOR_PATH} \
        --classifier {CLASSIFIER_PATH} \
        --output_video {OUTPUT_VIDEO} \
        --conf 0.35 \
        --classify_every 4
else:
    print('⚠️  Missing detector, classifier, or input video. Check paths above.')

## ➕ Cell 13 — Add More Videos (v11, v12, …) & Retrain
> When you're ready to expand beyond the first 10 videos:
>
> 1. Upload new video+XML to `data/v11/`, `data/v12/`, etc. on Drive
> 2. Re-run Cells 4–11 — they auto-discover new version folders
> 3. All new experiment metrics are appended to the log — Cell 11 will compare them

In [ ]:
# Check what's discovered now (re-run after adding new version folders)
exec(open('scripts/00_setup_drive.py').read())
print(f'Total version folders: {len(VERSION_FOLDERS)}')
print(f'Folders: {[v.name for v in VERSION_FOLDERS]}')